# 🃏 Arte das cartas — SDXL no Colab (T4)

**Antes de tudo:** menu `Ambiente de execução` → `Alterar o tipo de ambiente de execução` → **GPU T4**.

Depois é só rodar as células de cima pra baixo. A única que você mexe de verdade é a **célula 5 (PROMPTS)**.

Ordem: `1 GPU` → `2 Instalar` → `3 Onde salvar` → `4 Modelo` → `5 Estilo` → `6 Prompts` → `7 Gerar` 🎨

## 1. Conferir a GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "Sem GPU! Ambiente de execução → Alterar tipo → GPU T4"
print("\nGPU ok:", torch.cuda.get_device_name(0))

## 2. Instalar as bibliotecas

Leva ~1 minuto. Se aparecer um aviso pedindo pra reiniciar a sessão, **ignore** — não precisa.

In [ ]:
!pip -q install --upgrade diffusers transformers accelerate safetensors
print("pronto ✅")

## 3. Onde salvar as imagens

⚠️ **O Colab apaga tudo quando a sessão fecha.** Deixe `SALVAR_NO_DRIVE = True` pra tudo cair
numa pasta `card-game-arte` no seu Google Drive e não perder nada.

In [ ]:
import os

SALVAR_NO_DRIVE = True   # False = salva só no Colab (some ao fechar)

if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_SAIDA = '/content/drive/MyDrive/card-game-arte'
else:
    PASTA_SAIDA = '/content/card-game-arte'

os.makedirs(PASTA_SAIDA, exist_ok=True)
print("Salvando em:", PASTA_SAIDA)

## 4. Carregar o modelo

**Juggernaut XL v9** — SDXL treinado pra fantasia semi-realista, o melhor custo/benefício no T4.
O primeiro download leva ~3-5 min; depois fica em cache enquanto a sessão estiver viva.

Quer testar outro estilo? Troque a linha `MODELO`. Sugestões que também rodam no T4:
- `RunDiffusion/Juggernaut-XL-v9` → fantasia semi-realista (padrão)
- `stabilityai/stable-diffusion-xl-base-1.0` → SDXL puro, mais neutro
- `Lykon/dreamshaper-xl-1-0` → mais ilustrado/estilizado

In [ ]:
import torch, gc
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, DPMSolverMultistepScheduler

MODELO = "RunDiffusion/Juggernaut-XL-v9"

# VAE corrigido pra fp16 — sem isso o T4 às vezes cospe imagem preta
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)

def carregar(nome):
    try:
        return StableDiffusionXLPipeline.from_pretrained(
            nome, vae=vae, torch_dtype=torch.float16,
            variant="fp16", use_safetensors=True)
    except Exception:
        print("(sem variante fp16 nesse repo, baixando a padrão...)")
        return StableDiffusionXLPipeline.from_pretrained(
            nome, vae=vae, torch_dtype=torch.float16, use_safetensors=True)

pipe = carregar(MODELO)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, algorithm_type="dpmsolver++", use_karras_sigmas=True)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.set_progress_bar_config(disable=False)

gc.collect(); torch.cuda.empty_cache()
print("\nModelo pronto ✅ →", MODELO)

## 5. Estilo da coleção

Isso é o que gruda em **todos** os prompts e faz as 80 cartas parecerem do mesmo jogo.
Mexa aqui quando quiser mudar o visual geral (e aí regere tudo com o mesmo estilo).

⚠️ **Limite de 77 tokens:** o SDXL corta prompts muito longos. Some prompt + estilo e tente
ficar abaixo de ~60 palavras no total. Se cortar, o final do prompt simplesmente é ignorado.

In [ ]:
ESTILO = ("digital painting, fantasy trading card art, dramatic rim lighting, "
          "painterly brushwork, rich colors, centered character, "
          "simple atmospheric background, highly detailed")

NEGATIVO = ("text, watermark, signature, logo, ui, border, frame, "
            "blurry, lowres, jpeg artifacts, bad anatomy, deformed hands, "
            "extra fingers, extra limbs, cropped head, out of frame")

LARGURA, ALTURA   = 832, 1216   # proporção de carta (2:3). 1024x1024 se quiser quadrado
STEPS             = 30          # 25-35 é o ponto doce. Mais que isso quase não muda
CFG               = 6.0         # 5-7. Mais alto = obedece mais o prompt, porém mais 'duro'
IMAGENS_POR_PROMPT = 2          # quantas variações por carta

print("Estilo configurado ✅")

## 6. 🎯 SEUS PROMPTS — é aqui que você mexe

Cole os prompts da classe Tank aqui. O **nome da esquerda vira o nome do arquivo**,
então use algo tipo `tank_01_titã` pra achar fácil depois.

Dica: comece com 3-4 prompts só, veja se o estilo agradou, e só então cole os 20.

In [ ]:
PROMPTS = {

    # "nome_do_arquivo": "o prompt da carta",

    "tank_01": "a stout dwarven shieldbearer in heavy bronze armor, huge tower shield, braced stance",

    # "tank_02": "cole aqui",
    # "tank_03": "cole aqui",
    # "tank_04": "cole aqui",

}

print(f"{len(PROMPTS)} prompt(s) na fila → {IMAGENS_POR_PROMPT} imagem(ns) cada = {len(PROMPTS)*IMAGENS_POR_PROMPT} no total")

## 7. Gerar 🎨

~20-30 segundos por imagem no T4. Cada imagem aparece aqui embaixo **com a seed no nome** —
guarde a seed das que você gostar, dá pra reproduzir exatamente depois.

In [ ]:
import random, os, torch
from IPython.display import display

def gerar(prompts=None, seeds=None, quantas=None, mostrar=True):
    """prompts: dict {nome: prompt}. seeds: lista de seeds fixas (opcional)."""
    prompts = PROMPTS if prompts is None else prompts
    quantas = IMAGENS_POR_PROMPT if quantas is None else quantas
    feitas = []

    for nome, corpo in prompts.items():
        prompt_final = f"{corpo}, {ESTILO}"
        for i in range(quantas):
            seed = seeds[i % len(seeds)] if seeds else random.randint(0, 2**31 - 1)
            g = torch.Generator("cuda").manual_seed(seed)

            img = pipe(prompt=prompt_final,
                       negative_prompt=NEGATIVO,
                       width=LARGURA, height=ALTURA,
                       num_inference_steps=STEPS,
                       guidance_scale=CFG,
                       generator=g).images[0]

            caminho = os.path.join(PASTA_SAIDA, f"{nome}__seed{seed}.png")
            img.save(caminho)
            feitas.append(caminho)

            print(f"✅ {nome}   seed = {seed}")
            if mostrar:
                display(img.resize((LARGURA // 2, ALTURA // 2)))

            torch.cuda.empty_cache()

    print(f"\n{len(feitas)} imagem(ns) salva(s) em {PASTA_SAIDA}")
    return feitas


gerar()

---
## 8. Refazer só uma carta

Quando uma sair torta e você não quiser regerar tudo.

In [ ]:
# mais 4 variações de um prompt específico
gerar({"tank_01": PROMPTS["tank_01"]}, quantas=4)

# — ou — repetir uma seed que você gostou, mudando só o prompt:
# gerar({"tank_01_v2": "mesmo prompt com um detalhe trocado"}, seeds=[123456789], quantas=1)

## 9. (Opcional) Melhorar a resolução de uma favorita

Passa a imagem escolhida por um refino em 1.5x — dá mais detalhe de textura sem mudar a
composição. Use só nas que você já aprovou, porque é mais lento (~1 min).

In [ ]:
from diffusers import StableDiffusionXLImg2ImgPipeline
from PIL import Image

refino = StableDiffusionXLImg2ImgPipeline(**pipe.components)
refino.enable_attention_slicing()
refino.enable_vae_tiling()

def melhorar(caminho, prompt="", escala=1.5, forca=0.30, seed=0):
    base = Image.open(caminho).convert("RGB")
    w = int(base.width * escala) // 8 * 8
    h = int(base.height * escala) // 8 * 8
    base = base.resize((w, h), Image.LANCZOS)

    out = refino(prompt=f"{prompt}, {ESTILO}",
                 negative_prompt=NEGATIVO,
                 image=base, strength=forca,
                 num_inference_steps=STEPS, guidance_scale=CFG,
                 generator=torch.Generator("cuda").manual_seed(seed)).images[0]

    destino = caminho.replace(".png", "_HD.png")
    out.save(destino)
    print("salvo:", destino, out.size)
    display(out.resize((out.width // 2, out.height // 2)))
    torch.cuda.empty_cache()
    return destino


# melhorar(f"{PASTA_SAIDA}/tank_01__seed123456789.png", prompt=PROMPTS["tank_01"])

## 10. Baixar tudo num zip

(Se você marcou `SALVAR_NO_DRIVE = True`, já está tudo no Drive e isso aqui é dispensável.)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/cartas', 'zip', PASTA_SAIDA)
files.download('/content/cartas.zip')

---
### 🩹 Se der problema

| Sintoma | O que fazer |
|---|---|
| `CUDA out of memory` | `Ambiente de execução → Reiniciar sessão`, rode tudo de novo. Persistiu? Adicione `pipe.enable_model_cpu_offload()` depois do `.to("cuda")` (mais lento, gasta menos VRAM) |
| Imagem toda preta | O VAE fp16 não carregou — rode a célula 4 de novo |
| Prompt parece ignorado no final | Passou dos 77 tokens; encurte o prompt ou o `ESTILO` |
| Sessão caiu do nada | Limite do Colab grátis (~12h/dia). As imagens estão salvas no Drive 😉 |
| Mãos/dedos horríveis | Clássico do SDXL — evite prompts com "segurando", prefira arma apoiada/nas costas, ou aumente `IMAGENS_POR_PROMPT` e escolha |